### Portfolio Optimization 

This notebook implements a full portfolio analytics pipeline:
- Downloads asset prices (Yahoo Finance) or loads from CSV
- Computes log-returns and descriptive statistics
- Builds covariance/correlation structures
- Solves for multiple portfolios (min-variance, max-return, risk parity, tangency, etc.)
- Simulates an efficient frontier
- Generates plots and exports tables

In [1]:
# -----------------------------------------------------------------------------
# Select "fina6339" Python environment to run the script
# -----------------------------------------------------------------------------

# Parameters
DATA_SOURCE = "yfinance"   # "yfinance" or "csv"
CSV_PATH = None            
OUTPUT_DIR = "outputs"

START_DATE = "2020-01-01"
END_DATE = "2023-01-01"
PRICE_FIELD = "Adj Close"
AUTO_ADJUST = False
RISK_FREE_RATE = 0.02

TICKER_MAP = {
    "NIKE": "NKE",
    "AMZN": "AMZN",
    "GOOGLE": "GOOGL",
    "TESLA": "TSLA",
    "MSFT": "MSFT",
}

# Optional: set a random seed for reproducibility
RANDOM_SEED = 42

### Imports

In [2]:
import argparse
import inspect
import math
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import gaussian_kde

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from curl_cffi import requests as curl_requests
except Exception:
    curl_requests = None

In [3]:
# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

DEFAULT_TICKER_MAP: Dict[str, str] = {
    "NIKE": "NKE",
    "AMZN": "AMZN",
    "GOOGLE": "GOOGL",
    "TESLA": "TSLA",
    "MSFT": "MSFT",
}


# -----------------------------------------------------------------------------
# Data structures
# -----------------------------------------------------------------------------

@dataclass
class PortfolioResult:
    name: str
    weights: pd.Series
    returns: pd.Series
    mean: float
    std: float
    variance: float
    sharpe: Optional[float] = None
    metadata: Optional[Dict[str, float]] = None

In [4]:
# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def parse_version(version_string: str) -> Tuple[int, ...]:
    """Convert version text like '0.2.54' or '1.2.0' into a comparable tuple."""
    parts = re.findall(r"\d+", str(version_string))
    return tuple(int(p) for p in parts[:4]) if parts else (0,)

YFINANCE_VERSION = parse_version(getattr(yf, "__version__", "0")) if yf is not None else (0,)


def make_yfinance_session():
    """Create a curl-backed session when supported. This can help with Yahoo blocking."""
    if curl_requests is None:
        return None
    try:
        return curl_requests.Session(impersonate="chrome")
    except Exception:
        return None


def supported_kwargs(func, kwargs: Dict[str, object]) -> Dict[str, object]:
    """Keep only keyword arguments accepted by the function signature."""
    try:
        signature = inspect.signature(func)
        return {k: v for k, v in kwargs.items() if k in signature.parameters}
    except Exception:
        return kwargs


def resolve_price_field(price_field: str, auto_adjust: bool) -> str:
    """
    Newer yfinance versions often omit 'Adj Close' when auto_adjust=True.
    In that case, use 'Close'.
    """
    if auto_adjust and price_field == "Adj Close":
        return "Close"
    return price_field


def annualized_sharpe(returns: pd.Series, risk_free_rate: float = 0.0, periods: int = 252) -> float:
    clean = pd.Series(returns).dropna()
    if clean.empty:
        return np.nan
    excess = clean.mean() - risk_free_rate
    sigma = clean.std(ddof=1)
    if sigma <= 0:
        return np.nan
    return float(np.sqrt(periods) * excess / sigma)


def ensure_nonempty(df: pd.DataFrame, context: str) -> None:
    if df is None or df.empty:
        raise ValueError(f"No data available for {context}.")

In [5]:
# -----------------------------------------------------------------------------
# Data loading and processing
# -----------------------------------------------------------------------------

def compute_log_returns_from_prices(prices: pd.DataFrame) -> pd.DataFrame:
    prices = prices.sort_index().dropna(how="all")
    returns = np.log(prices / prices.shift(1))
    returns = returns.replace([np.inf, -np.inf], np.nan).dropna(how="any")
    returns.index = pd.to_datetime(returns.index)
    returns.index.name = "Date"
    return returns


def extract_price_frame(raw: pd.DataFrame, tickers: List[str], price_field: str) -> pd.DataFrame:
    """Extract a price matrix from either a flat-column or MultiIndex yfinance download."""
    ensure_nonempty(raw, "Yahoo Finance raw download")

    if isinstance(raw.columns, pd.MultiIndex):
        level0 = list(raw.columns.get_level_values(0))
        if price_field in level0:
            prices = raw[price_field].copy()
        elif "Close" in level0:
            prices = raw["Close"].copy()
        else:
            available = sorted(set(level0))
            raise ValueError(
                f"Price field '{price_field}' was not found in the download. Available fields: {available}"
            )
    else:
        if price_field in raw.columns:
            prices = raw[[price_field]].copy()
        elif "Close" in raw.columns:
            prices = raw[["Close"]].copy()
        else:
            available = list(raw.columns)
            raise ValueError(
                f"Price field '{price_field}' was not found in the download. Available fields: {available}"
            )
        prices.columns = tickers[:1]

    prices.index = pd.to_datetime(prices.index)
    prices.index.name = "Date"
    return prices.sort_index()


def download_prices_yfinance(
    ticker_map: Dict[str, str],
    start: str,
    end: str,
    price_field: str = "Adj Close",
    auto_adjust: bool = False,
    pause: float = 1.0,
    max_retries: int = 3,
) -> pd.DataFrame:
    """
    Yahoo Finance downloader.

    1. Attempt one batch download with conservative settings.
    2. If that fails or returns incomplete results, download each ticker separately.
    3. Keep only supported keyword arguments to remain version-compatible.
    """
    if yf is None:
        raise ImportError("yfinance is not installed. Install it with: pip install yfinance")

    session = make_yfinance_session()
    symbols = list(ticker_map.values())
    labels_by_symbol = {symbol: label for label, symbol in ticker_map.items()}
    resolved_field = resolve_price_field(price_field, auto_adjust)

    batch_kwargs = {
        "tickers": symbols,
        "start": start,
        "end": end,
        "progress": False,
        "threads": False,
        "group_by": "column",
        "auto_adjust": auto_adjust,
        "repair": True,
        "timeout": 30,
        "ignore_tz": True,
        "session": session,
    }
    batch_kwargs = supported_kwargs(yf.download, batch_kwargs)

    batch_error = None
    batch_prices = pd.DataFrame()
    try:
        raw = yf.download(**batch_kwargs)
        batch_prices = extract_price_frame(raw, symbols, resolved_field)
    except Exception as exc:
        batch_error = exc

    expected = set(symbols)
    received = set(batch_prices.columns) if not batch_prices.empty else set()
    missing = sorted(expected - received)

    if not batch_prices.empty and not missing:
        prices = batch_prices.copy()
    else:
        series_list: List[pd.Series] = []
        failures: Dict[str, str] = {}

        for symbol in symbols:
            ticker_obj = None
            last_error = None

            for attempt in range(1, max_retries + 1):
                try:
                    ticker_kwargs = {"ticker": symbol, "session": session}
                    ticker_kwargs = supported_kwargs(yf.Ticker, ticker_kwargs)
                    if "ticker" in ticker_kwargs:
                        ticker_obj = yf.Ticker(**ticker_kwargs)
                    else:
                        ticker_obj = yf.Ticker(symbol)

                    history_kwargs = {
                        "start": start,
                        "end": end,
                        "interval": "1d",
                        "auto_adjust": auto_adjust,
                        "repair": True,
                        "timeout": 30,
                        "raise_errors": True,
                    }
                    history_kwargs = supported_kwargs(ticker_obj.history, history_kwargs)
                    hist = ticker_obj.history(**history_kwargs)
                    ensure_nonempty(hist, f"Yahoo Finance history for {symbol}")

                    candidate_field = resolved_field
                    if candidate_field not in hist.columns and candidate_field == "Adj Close" and "Close" in hist.columns:
                        candidate_field = "Close"
                    if candidate_field not in hist.columns:
                        raise ValueError(
                            f"Field '{candidate_field}' not found for {symbol}. Available columns: {list(hist.columns)}"
                        )

                    series = hist[candidate_field].copy()
                    series.name = symbol
                    series_list.append(series)
                    last_error = None
                    break
                except Exception as exc:
                    last_error = exc
                    time.sleep(pause * attempt)

            if last_error is not None:
                failures[symbol] = str(last_error)

        if not series_list:
            message = "No data was downloaded from Yahoo Finance."
            if batch_error is not None:
                message += f" Batch error: {batch_error}."
            if failures:
                message += f" Per-ticker failures: {failures}."
            raise ValueError(message)

        prices = pd.concat(series_list, axis=1).sort_index()
        if failures:
            print("Warning: some ticker downloads failed:")
            for symbol, err in failures.items():
                print(f"  - {symbol}: {err}")

    prices = prices.rename(columns=labels_by_symbol)
    ordered_cols = [label for label in ticker_map.keys() if label in prices.columns]
    missing_labels = [label for label in ticker_map.keys() if label not in prices.columns]
    if missing_labels:
        raise ValueError(
            "Missing downloaded series for these assets: "
            f"{missing_labels}. This usually means Yahoo throttled the request or the environment blocked access."
        )

    prices = prices[ordered_cols].dropna(how="all")
    ensure_nonempty(prices, "final Yahoo Finance price matrix")
    return prices


def load_returns_csv(csv_path: str | Path, assets: Optional[Iterable[str]] = None) -> pd.DataFrame:
    csv_path = Path(csv_path)
    df = pd.read_csv(csv_path)
    df.columns = [str(c).strip() for c in df.columns]

    date_col = next((c for c in ["Date", "date", "DATE"] if c in df.columns), None)
    if date_col is not None:
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.set_index(date_col)
    else:
        df.index = pd.bdate_range("2020-01-01", periods=len(df))

    if assets is None:
        keep_cols = list(df.columns)
    else:
        keep_cols = [c for c in assets if c in df.columns]

    if not keep_cols:
        raise ValueError("No matching asset columns were found in the CSV.")

    returns = df[keep_cols].astype(float)
    returns = returns.replace([np.inf, -np.inf], np.nan).dropna(how="any")
    returns.index.name = "Date"
    ensure_nonempty(returns, "CSV returns")
    return returns


def load_data(
    source: str = "yfinance",
    csv_path: Optional[str | Path] = None,
    ticker_map: Optional[Dict[str, str]] = None,
    start: str = "2020-01-01",
    end: str = "2023-01-01",
    price_field: str = "Adj Close",
    auto_adjust: bool = False,
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    ticker_map = ticker_map or DEFAULT_TICKER_MAP.copy()
    source = source.lower().strip()

    if source == "yfinance":
        prices = download_prices_yfinance(
            ticker_map=ticker_map,
            start=start,
            end=end,
            price_field=price_field,
            auto_adjust=auto_adjust,
        )
        returns = compute_log_returns_from_prices(prices)
        return returns, prices

    if source == "csv":
        if csv_path is None:
            raise ValueError("csv_path must be supplied when source='csv'.")
        returns = load_returns_csv(csv_path=csv_path, assets=ticker_map.keys())
        return returns, None

    raise ValueError("source must be either 'yfinance' or 'csv'.")

### Optimization functions

In [6]:
# -----------------------------------------------------------------------------
# Portfolio mathematics
# -----------------------------------------------------------------------------

def portfolio_return(weights: np.ndarray, mean_returns: np.ndarray) -> float:
    return float(weights @ mean_returns)


def portfolio_variance(weights: np.ndarray, cov_matrix: np.ndarray) -> float:
    return float(weights @ cov_matrix @ weights)


def portfolio_std(weights: np.ndarray, cov_matrix: np.ndarray) -> float:
    return math.sqrt(max(portfolio_variance(weights, cov_matrix), 0.0))


def portfolio_returns_series(returns: pd.DataFrame, weights: pd.Series) -> pd.Series:
    aligned = weights.reindex(returns.columns).fillna(0.0)
    portfolio = returns.mul(aligned, axis=1).sum(axis=1)
    portfolio.name = "portfolio_return"
    return portfolio


def descriptive_statistics(series: pd.Series) -> Dict[str, float]:
    clean = pd.Series(series).dropna()
    return {
        "count": float(clean.count()),
        "min": float(clean.min()),
        "max": float(clean.max()),
        "range": float(clean.max() - clean.min()),
        "sum": float(clean.sum()),
        "median": float(clean.median()),
        "mean": float(clean.mean()),
        "std": float(clean.std(ddof=1)),
        "var": float(clean.var(ddof=1)),
        "se_mean": float(clean.std(ddof=1) / math.sqrt(len(clean))) if len(clean) else np.nan,
        "coef_var": float(clean.std(ddof=1) / clean.mean()) if clean.mean() != 0 else np.nan,
        "skew": float(clean.skew()),
        "kurtosis": float(clean.kurt()),
    }


def asset_summary_table(returns: pd.DataFrame) -> pd.DataFrame:
    table = pd.DataFrame({c: descriptive_statistics(returns[c]) for c in returns.columns}).T
    table.index.name = "asset"
    return table


def portfolio_summary_table(portfolios: List[PortfolioResult], risk_free_rate: float = 0.0) -> pd.DataFrame:
    rows = []
    for p in portfolios:
        rows.append(
            {
                "portfolio": p.name,
                "mean": p.mean,
                "std": p.std,
                "variance": p.variance,
                "daily_sharpe": (p.mean - risk_free_rate) / p.std if p.std > 0 else np.nan,
                "annualized_sharpe": annualized_sharpe(p.returns, risk_free_rate=risk_free_rate),
            }
        )
    return pd.DataFrame(rows).set_index("portfolio")

In [7]:
# -----------------------------------------------------------------------------
# Optimization 
# -----------------------------------------------------------------------------

def optimize_min_variance(returns: pd.DataFrame) -> PortfolioResult:
    cov = returns.cov().values
    n = returns.shape[1]
    x0 = np.repeat(1.0 / n, n)
    bounds = [(-1.0, 1.0)] * n
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

    result = minimize(
        lambda w: portfolio_variance(w, cov),
        x0=x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
    )
    if not result.success:
        raise RuntimeError(f"Minimum-variance optimization failed: {result.message}")

    weights = pd.Series(result.x, index=returns.columns, name="weight")
    port = portfolio_returns_series(returns, weights)
    return PortfolioResult(
        name="Min-variance",
        weights=weights,
        returns=port,
        mean=float(port.mean()),
        std=float(port.std(ddof=1)),
        variance=float(port.var(ddof=1)),
    )


def optimize_max_return_box(returns: pd.DataFrame, min_w: float = 0.0, max_w: float = 0.25) -> PortfolioResult:
    mu = returns.mean().values
    n = returns.shape[1]
    x0 = np.repeat(1.0 / n, n)
    bounds = [(min_w, max_w)] * n
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

    result = minimize(
        lambda w: -portfolio_return(w, mu),
        x0=x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
    )
    if not result.success:
        raise RuntimeError(f"Maximum-return optimization failed: {result.message}")

    weights = pd.Series(result.x, index=returns.columns, name="weight")
    port = portfolio_returns_series(returns, weights)
    return PortfolioResult(
        name="Max-return",
        weights=weights,
        returns=port,
        mean=float(port.mean()),
        std=float(port.std(ddof=1)),
        variance=float(port.var(ddof=1)),
    )


def random_dollar_neutral_portfolio(
    returns: pd.DataFrame,
    target_mean: float = 0.00015,
    target_std: float = 0.0002,
    max_positions: int = 3,
    bound: float = 0.25,
    n_samples: int = 25000,
    random_state: int = 42,
) -> PortfolioResult:
    rng = np.random.default_rng(random_state)
    assets = returns.columns.tolist()
    best_score = np.inf
    best_w = None

    for _ in range(n_samples):
        k = int(rng.integers(1, max_positions + 1))
        idx = rng.choice(len(assets), size=k, replace=False)
        w = np.zeros(len(assets))

        raw = rng.uniform(-bound, bound, size=k)
        raw = raw - raw.mean()
        raw = np.clip(raw, -bound, bound)
        w[idx] = raw

        if abs(np.sum(w)) > 0.01:
            continue
        if np.count_nonzero(np.abs(w) > 1e-12) > max_positions:
            continue

        port = returns.values @ w
        mu = float(np.mean(port))
        sigma = float(np.std(port, ddof=1))
        score = (mu - target_mean) ** 2 + (sigma - target_std) ** 2

        if score < best_score:
            best_score = score
            best_w = w.copy()

    if best_w is None:
        raise RuntimeError("No feasible dollar-neutral portfolio was found.")

    weights = pd.Series(best_w, index=assets, name="weight")
    port = portfolio_returns_series(returns, weights)
    return PortfolioResult(
        name="Dollar-neutral random search",
        weights=weights,
        returns=port,
        mean=float(port.mean()),
        std=float(port.std(ddof=1)),
        variance=float(port.var(ddof=1)),
        metadata={
            "target_mean": target_mean,
            "target_std": target_std,
            "max_positions": float(max_positions),
            "weight_bound": float(bound),
            "samples": float(n_samples),
        },
    )


def optimize_risk_parity(returns: pd.DataFrame) -> PortfolioResult:
    cov = returns.cov().values
    n = returns.shape[1]


    def risk_contributions(w: np.ndarray) -> np.ndarray:
        port_var = portfolio_variance(w, cov)
        mrc = cov @ w
        return w * mrc / max(port_var, 1e-15)


    def objective(w: np.ndarray) -> float:
        rc = risk_contributions(w)
        target = np.repeat(1.0 / n, n)
        return float(np.sum((rc - target) ** 2))

    x0 = np.repeat(1.0 / n, n)
    bounds = [(0.0, 1.0)] * n
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

    result = minimize(objective, x0=x0, method="SLSQP", bounds=bounds, constraints=constraints)
    if not result.success:
        raise RuntimeError(f"Risk-parity optimization failed: {result.message}")

    weights = pd.Series(result.x, index=returns.columns, name="weight")
    port = portfolio_returns_series(returns, weights)
    return PortfolioResult(
        name="Risk-parity",
        weights=weights,
        returns=port,
        mean=float(port.mean()),
        std=float(port.std(ddof=1)),
        variance=float(port.var(ddof=1)),
    )


def optimize_tangency_portfolio(returns: pd.DataFrame, risk_free_rate: float = 0.0) -> PortfolioResult:
    mu = returns.mean().values
    cov = returns.cov().values
    n = returns.shape[1]


    def objective(w: np.ndarray) -> float:
        sigma = portfolio_std(w, cov)
        if sigma <= 1e-12:
            return 1e6
        sharpe = (portfolio_return(w, mu) - risk_free_rate) / sigma
        return -sharpe

    x0 = np.repeat(1.0 / n, n)
    bounds = [(0.0, 1.0)] * n
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

    result = minimize(objective, x0=x0, method="SLSQP", bounds=bounds, constraints=constraints)
    if not result.success:
        raise RuntimeError(f"Tangency optimization failed: {result.message}")

    weights = pd.Series(result.x, index=returns.columns, name="weight")
    port = portfolio_returns_series(returns, weights)
    sigma = float(port.std(ddof=1))
    sharpe = (float(port.mean()) - risk_free_rate) / sigma if sigma > 0 else np.nan
    return PortfolioResult(
        name="Tangency",
        weights=weights,
        returns=port,
        mean=float(port.mean()),
        std=sigma,
        variance=float(port.var(ddof=1)),
        sharpe=float(sharpe),
    )


def simulate_efficient_frontier(
    returns: pd.DataFrame,
    n_portfolios: int = 5000,
    random_state: int = 42,
    long_only: bool = True,
    risk_free_rate: float = 0.0,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    mu = returns.mean().values
    cov = returns.cov().values
    n_assets = returns.shape[1]

    rows = []
    for _ in range(n_portfolios):
        if long_only:
            w = rng.random(n_assets)
            w = w / w.sum()
        else:
            w = rng.normal(size=n_assets)
            if np.sum(np.abs(w)) == 0:
                continue
            w = w / np.sum(np.abs(w))

        port_mu = portfolio_return(w, mu)
        port_sigma = portfolio_std(w, cov)
        sharpe = (port_mu - risk_free_rate) / port_sigma if port_sigma > 0 else np.nan
        row = {"mean": port_mu, "std": port_sigma, "sharpe": sharpe}
        row.update({asset: weight for asset, weight in zip(returns.columns, w)})
        rows.append(row)

    return pd.DataFrame(rows)

In [8]:
# -----------------------------------------------------------------------------
# Plotting
# -----------------------------------------------------------------------------

def savefig(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()


def plot_prices(prices: pd.DataFrame, output_path: Path) -> None:
    plt.figure(figsize=(12, 5.5))
    for col in prices.columns:
        plt.plot(prices.index, prices[col], label=col, linewidth=1.1)
    plt.title("Asset prices (source: Yahoo Finance)")
    # plt.xlabel("Date")
    plt.ylabel("Price (in $)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    savefig(output_path)


def plot_asset_returns(returns: pd.DataFrame, output_path: Path) -> None:
    plt.figure(figsize=(12, 5.5))
    for col in returns.columns:
        plt.plot(returns.index, returns[col], label=col, linewidth=1.0)
    plt.title("Daily log-returns")
    # plt.xlabel("Date")
    plt.ylabel("Log-returns (in decimal)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    savefig(output_path)


def plot_cumulative_returns(returns: pd.DataFrame, output_path: Path) -> None:
    cumulative = np.exp(returns.cumsum())
    plt.figure(figsize=(12, 5.5))
    for col in cumulative.columns:
        plt.plot(cumulative.index, cumulative[col], label=col, linewidth=1.1)
    plt.title("Cumulative returns from log-returns")
    # plt.xlabel("Date")
    plt.ylabel("Growth of $1")
    plt.grid(True, alpha=0.3)
    plt.legend()
    savefig(output_path)


def plot_histogram(series: pd.Series, title: str, output_path: Path, bins: int = 30) -> None:
    plt.figure(figsize=(9, 5))
    plt.hist(series, bins=bins, edgecolor="black", alpha=0.8)
    plt.title(title)
    plt.xlabel("Returns")
    plt.ylabel("Frequency")
    plt.grid(True, axis="y", alpha=0.3)
    savefig(output_path)


def plot_density(series: pd.Series, title: str, output_path: Path) -> None:
    clean = pd.Series(series).dropna()
    x = np.linspace(clean.min(), clean.max(), 400)
    kde = gaussian_kde(clean.values)
    y = kde(x)
    plt.figure(figsize=(9, 5))
    plt.plot(x, y, linewidth=2)
    plt.title(title)
    plt.xlabel("Returns")
    plt.ylabel("Density")
    plt.grid(True, alpha=0.3)
    savefig(output_path)


def plot_weights(portfolio: PortfolioResult, output_path: Path) -> None:
    plt.figure(figsize=(8.5, 4.8))
    portfolio.weights.plot(kind="bar")
    plt.title(f"{portfolio.name} portfolio weights")
    plt.xlabel("Assets")
    plt.ylabel("Weights")
    plt.grid(True, axis="y", alpha=0.3)
    savefig(output_path)


def plot_portfolio_comparison(portfolios: List[PortfolioResult], output_path: Path) -> None:
    plt.figure(figsize=(8.5, 5.5))
    means = [p.mean for p in portfolios]
    stds = [p.std for p in portfolios]
    labels = [p.name for p in portfolios]
    plt.scatter(stds, means, s=70)
    for x, y, label in zip(stds, means, labels):
        plt.annotate(label, (x, y), xytext=(6, 6), textcoords="offset points")
    plt.title("Portfolio risk-return comparison")
    plt.xlabel("Standard deviation")
    plt.ylabel("Mean return")
    plt.grid(True, alpha=0.3)
    savefig(output_path)


def plot_efficient_frontier(
    frontier: pd.DataFrame,
    portfolios: List[PortfolioResult],
    output_path: Path,
) -> None:
    plt.figure(figsize=(9, 5.8))
    plt.scatter(frontier["std"], frontier["mean"], s=10, alpha=0.35)
    for p in portfolios:
        plt.scatter([p.std], [p.mean], s=90)
        plt.annotate(p.name, (p.std, p.mean), xytext=(6, 6), textcoords="offset points")
    plt.title("Simulated efficient frontier and optimized portfolios")
    plt.xlabel("Standard deviation")
    plt.ylabel("Mean return")
    plt.grid(True, alpha=0.3)
    savefig(output_path)


def plot_indifference_curves(output_path: Path, risk_aversion: Optional[List[float]] = None) -> None:
    if risk_aversion is None:
        risk_aversion = [1, 2, 3, 4]
    sigma = np.linspace(0, 0.08, 250)
    plt.figure(figsize=(8.5, 5.5))
    for a in risk_aversion:
        utility_curve = 0.015 + 0.5 * a * sigma**2
        plt.plot(sigma, utility_curve, label=f"A = {a}")
    plt.title("Indifference curves")
    plt.xlabel("Portfolio risk (std. dev.)")
    plt.ylabel("Expected return for constant utility")
    plt.grid(True, alpha=0.3)
    plt.legend()
    savefig(output_path)

In [9]:
# -----------------------------------------------------------------------------
# Save tables
# -----------------------------------------------------------------------------

def save_table(df: pd.DataFrame, path: Path, float_format: str = "%.8f") -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path)
    with open(path.with_suffix(".md"), "w", encoding="utf-8") as handle:
        handle.write(df.to_markdown(floatfmt=float_format))

### Main analysis 

In [10]:
# -----------------------------------------------------------------------------
# Main analysis (pipeline)
# -----------------------------------------------------------------------------

def run_analysis(
    source: str = "yfinance",
    csv_path: Optional[str | Path] = None,
    output_dir: str | Path = "outputs",
    ticker_map: Optional[Dict[str, str]] = None,
    start: str = "2020-01-01",
    end: str = "2023-01-01",
    price_field: str = "Adj Close",
    auto_adjust: bool = False,
    risk_free_rate: float = 0.0,
    frontier_samples: int = 5000,
) -> Dict[str, object]:
    output_dir = Path(output_dir)
    figures_dir = output_dir / "figures"
    tables_dir = output_dir / "tables"
    output_dir.mkdir(parents=True, exist_ok=True)

    returns, prices = load_data(
        source=source,
        csv_path=csv_path,
        ticker_map=ticker_map or DEFAULT_TICKER_MAP,
        start=start,
        end=end,
        price_field=price_field,
        auto_adjust=auto_adjust,
    )

    asset_stats = asset_summary_table(returns)
    correlation = returns.corr()
    covariance = returns.cov()

    min_var = optimize_min_variance(returns)
    max_ret = optimize_max_return_box(returns)
    neutral = random_dollar_neutral_portfolio(returns)
    risk_parity = optimize_risk_parity(returns)
    tangency = optimize_tangency_portfolio(returns, risk_free_rate=risk_free_rate)
    portfolios = [min_var, max_ret, neutral, risk_parity, tangency]

    portfolio_stats = portfolio_summary_table(portfolios, risk_free_rate=risk_free_rate)
    weights_table = pd.concat([p.weights.rename(p.name) for p in portfolios], axis=1)
    frontier = simulate_efficient_frontier(
        returns,
        n_portfolios=frontier_samples,
        random_state=42,
        long_only=True,
        risk_free_rate=risk_free_rate,
    )

    if prices is not None:
        plot_prices(prices, figures_dir / "01_prices.png")
    plot_asset_returns(returns, figures_dir / "02_asset_returns.png")
    plot_cumulative_returns(returns, figures_dir / "03_cumulative_returns.png")

    for i, portfolio in enumerate(portfolios, start=1):
        slug = portfolio.name.lower().replace(" ", "_").replace("-", "_")
        plot_weights(portfolio, figures_dir / f"{10 + i:02d}_{slug}_weights.png")
        plot_histogram(portfolio.returns, f"{portfolio.name} portfolio return histogram", figures_dir / f"{20 + i:02d}_{slug}_histogram.png")
        plot_density(portfolio.returns, f"{portfolio.name} portfolio return density", figures_dir / f"{30 + i:02d}_{slug}_density.png")

    plot_portfolio_comparison(portfolios, figures_dir / "40_portfolio_risk_return_comparison.png")
    plot_efficient_frontier(frontier, portfolios, figures_dir / "41_efficient_frontier.png")
    plot_indifference_curves(figures_dir / "42_indifference_curves.png")

    save_table(asset_stats, tables_dir / "asset_summary_statistics.csv")
    save_table(correlation, tables_dir / "asset_correlation_matrix.csv")
    save_table(covariance, tables_dir / "asset_covariance_matrix.csv")
    save_table(portfolio_stats, tables_dir / "portfolio_summary_statistics.csv")
    save_table(weights_table, tables_dir / "portfolio_weights.csv")
    save_table(frontier, tables_dir / "simulated_efficient_frontier.csv")

    return {
        "returns": returns,
        "prices": prices,
        "asset_stats": asset_stats,
        "correlation": correlation,
        "covariance": covariance,
        "portfolio_stats": portfolio_stats,
        "weights": weights_table,
        "efficient_frontier": frontier,
        "portfolios": portfolios,
        "output_dir": output_dir,
        "yfinance_version": getattr(yf, "__version__", None),
    }

In [11]:
# -----------------------------------------------------------------------------
# Command-line interface
# -----------------------------------------------------------------------------

def main() -> None:
    parser = argparse.ArgumentParser(
        description="Portfolio optimization workflow with Yahoo Finance support."
    )
    parser.add_argument("--source", default="yfinance", choices=["yfinance", "csv"])
    parser.add_argument("--csv", default=None, help="Path to a CSV file when source=csv.")
    parser.add_argument("--output-dir", default="outputs")
    parser.add_argument("--start", default="2020-01-01")
    parser.add_argument("--end", default="2023-01-01")
    parser.add_argument("--price-field", default="Adj Close")
    parser.add_argument("--auto-adjust", action="store_true")
    parser.add_argument("--risk-free-rate", type=float, default=0.0)
    parser.add_argument("--frontier-samples", type=int, default=5000)
    args = parser.parse_args([] if "ipykernel" in sys.modules else None)

    results = run_analysis(
        source=args.source,
        csv_path=args.csv,
        output_dir=args.output_dir,
        start=args.start,
        end=args.end,
        price_field=args.price_field,
        auto_adjust=args.auto_adjust,
        risk_free_rate=args.risk_free_rate,
        frontier_samples=args.frontier_samples,
    )

    print("Analysis completed successfully.")
    print(f"yfinance version: {results['yfinance_version']}")
    print(f"Output directory: {results['output_dir']}")


if __name__ == "__main__":
    if "ipykernel" not in sys.modules:
        main()

### Run script

In [12]:
# Run the full analysis
results = run_analysis(
    source=DATA_SOURCE,
    csv_path=CSV_PATH,
    output_dir=OUTPUT_DIR,
    ticker_map=TICKER_MAP,
    start=START_DATE,
    end=END_DATE,
    price_field=PRICE_FIELD,
    auto_adjust=AUTO_ADJUST,
    risk_free_rate=RISK_FREE_RATE,
)

print(f"Saved outputs to: {results['output_dir']}")
try:
    print("Available result keys:", sorted(results.keys()))
except Exception:
    pass

Saved outputs to: outputs
Available result keys: ['asset_stats', 'correlation', 'covariance', 'efficient_frontier', 'output_dir', 'portfolio_stats', 'portfolios', 'prices', 'returns', 'weights', 'yfinance_version']
